In [1]:
import pandas as pd
import numpy as np
import sklearn
import scipy
import lightgbm
import shap
import matplotlib

print("Environment setup OK")

Matplotlib is building the font cache; this may take a moment.


Environment setup OK


In [2]:
from pathlib import Path

DATA_DIR = Path("../data/mock")

print("Data folder:", DATA_DIR.resolve())
print("Exists:", DATA_DIR.exists())

Data folder: E:\J26-DS-349\component4\data\mock
Exists: True


In [3]:
products = pd.read_csv(DATA_DIR / "products.csv")
preferences = pd.read_csv(DATA_DIR / "user_preferences.csv")
c1 = pd.read_csv(DATA_DIR / "c1_product_aspects.csv")
c2 = pd.read_csv(DATA_DIR / "c2_review_evidence.csv")
c3 = pd.read_csv(DATA_DIR / "c3_temporal_features.csv")

print("Products:", products.shape)
print("Preferences:", preferences.shape)
print("C1:", c1.shape)
print("C2:", c2.shape)
print("C3:", c3.shape)

Products: (5, 4)
Preferences: (6, 5)
C1: (30, 5)
C2: (90, 5)
C3: (30, 6)


In [4]:
datasets = {
    "products": products,
    "preferences": preferences,
    "c1": c1,
    "c2": c2,
    "c3": c3
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


products
product_id      0
product_name    0
category        0
price           0
dtype: int64

preferences
user_id       0
aspect        0
importance    0
confidence    0
max_budget    0
dtype: int64

c1
product_id         0
aspect             0
aspect_quality     0
sentiment_score    0
c1_confidence      0
dtype: int64

c2
review_id                  0
product_id                 0
aspect                     0
evidence_quality           0
representative_evidence    0
dtype: int64

c3
product_id              0
aspect                  0
recent_quality          0
trend_score             0
temporal_uncertainty    0
emerging_issue_score    0
dtype: int64


In [5]:
valid_products = set(products["product_id"])

print("C1 invalid product IDs:",
      set(c1["product_id"]) - valid_products)

print("C2 invalid product IDs:",
      set(c2["product_id"]) - valid_products)

print("C3 invalid product IDs:",
      set(c3["product_id"]) - valid_products)

C1 invalid product IDs: set()
C2 invalid product IDs: set()
C3 invalid product IDs: set()


In [6]:
pref_aspects = set(preferences["aspect"])
c1_aspects = set(c1["aspect"])
c2_aspects = set(c2["aspect"])
c3_aspects = set(c3["aspect"])

print("Preference aspects:", pref_aspects)
print("C1 matches preferences:", c1_aspects == pref_aspects)
print("C2 matches preferences:", c2_aspects == pref_aspects)
print("C3 matches preferences:", c3_aspects == pref_aspects)

Preference aspects: {'performance', 'durability', 'display', 'battery', 'price', 'camera'}
C1 matches preferences: True
C2 matches preferences: True
C3 matches preferences: True


In [7]:
checks = {
    "Importance between 1 and 5":
        preferences["importance"].between(1, 5).all(),

    "Preference confidence between 0 and 1":
        preferences["confidence"].between(0, 1).all(),

    "C1 aspect quality between 0 and 1":
        c1["aspect_quality"].between(0, 1).all(),

    "C1 sentiment score between 0 and 1":
        c1["sentiment_score"].between(0, 1).all(),

    "C1 confidence between 0 and 1":
        c1["c1_confidence"].between(0, 1).all(),

    "C2 evidence quality between 0 and 1":
        c2["evidence_quality"].between(0, 1).all(),

    "C3 recent quality between 0 and 1":
        c3["recent_quality"].between(0, 1).all(),

    "C3 temporal uncertainty between 0 and 1":
        c3["temporal_uncertainty"].between(0, 1).all(),

    "C3 emerging issue score between 0 and 1":
        c3["emerging_issue_score"].between(0, 1).all()
}

for test, passed in checks.items():
    print(f"{test}: {'PASS' if passed else 'FAIL'}")

Importance between 1 and 5: PASS
Preference confidence between 0 and 1: PASS
C1 aspect quality between 0 and 1: PASS
C1 sentiment score between 0 and 1: PASS
C1 confidence between 0 and 1: PASS
C2 evidence quality between 0 and 1: PASS
C3 recent quality between 0 and 1: PASS
C3 temporal uncertainty between 0 and 1: PASS
C3 emerging issue score between 0 and 1: PASS


In [8]:
print(
    "Duplicate C1 product-aspect rows:",
    c1.duplicated(subset=["product_id", "aspect"]).sum()
)

print(
    "Duplicate C3 product-aspect rows:",
    c3.duplicated(subset=["product_id", "aspect"]).sum()
)

print(
    "Duplicate review IDs:",
    c2["review_id"].duplicated().sum()
)

Duplicate C1 product-aspect rows: 0
Duplicate C3 product-aspect rows: 0
Duplicate review IDs: 0


In [9]:
all_valid = (
    all(checks.values())
    and len(set(c1["product_id"]) - valid_products) == 0
    and len(set(c2["product_id"]) - valid_products) == 0
    and len(set(c3["product_id"]) - valid_products) == 0
    and c1_aspects == pref_aspects
    and c2_aspects == pref_aspects
    and c3_aspects == pref_aspects
    and c1.duplicated(subset=["product_id", "aspect"]).sum() == 0
    and c3.duplicated(subset=["product_id", "aspect"]).sum() == 0
    and c2["review_id"].duplicated().sum() == 0
)

print("MOCK DATA VALIDATION:", "PASSED" if all_valid else "FAILED")

MOCK DATA VALIDATION: PASSED
